In [1]:
# enhanced_image_predictor_clean.py
# 단일 창 음식 이미지 예측기 - 깔끔한 버전

import os
import glob
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from PIL import Image, ImageTk
from io import BytesIO
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre
import time
import threading
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox, filedialog
from datetime import datetime
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import queue
warnings.filterwarnings('ignore')

# 기본 설정
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# GPU 설정
def setup_gpu():
    """GPU 설정 및 메모리 증가 허용"""
    try:
        # GPU 디바이스 확인
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            try:
                # GPU 메모리 증가 허용 (메모리 부족 방지)
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                
                # 첫 번째 GPU를 기본으로 설정
                tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
                
                print(f"[GPU INFO] GPU 사용 가능: {len(gpus)}개")
                print(f"[GPU INFO] 사용 중인 GPU: {gpus[0].name}")
                return True
            except RuntimeError as e:
                print(f"[GPU WARNING] GPU 설정 실패: {e}")
                return False
        else:
            print("[GPU INFO] GPU를 찾을 수 없습니다. CPU를 사용합니다.")
            return False
    except Exception as e:
        print(f"[GPU ERROR] GPU 초기화 오류: {e}")
        return False

# GPU 초기화
GPU_AVAILABLE = setup_gpu()

IMG_SIZE = (224, 224)

class FoodPredictorGUI:
    """단일 창 GUI 애플리케이션"""
    
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("음식 이미지 URL 연속 예측 시스템")
        self.root.geometry("1280x800")
        
        # 예측기
        self.predictor = None
        
        # 실행 상태
        self.is_running = False
        self.stop_requested = False
        
        # 현재 이미지 정보
        self.current_image = None
        self.current_results = None
        self.current_properties = None
        self.current_menu_name = None  # 시트의 실제 메뉴명
        self.user_action = None
        
        # UI 먼저 설정
        self.setup_ui()
        
        # UI 설정 후 매칭 테이블 로드
        self.load_translation_table()
        
    def setup_ui(self):
        """UI 설정"""
        # 메인 컨테이너
        main_container = tk.Frame(self.root)
        main_container.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # 상단: 제목 및 설정
        top_frame = tk.Frame(main_container)
        top_frame.pack(fill=tk.X, pady=(0, 10))
        
        # 제목
        title_label = tk.Label(top_frame, text="음식 이미지 URL 예측 시스템", 
                              font=('Arial', 18, 'bold'), fg='darkblue')
        title_label.pack(pady=5)
        
        # 설정 프레임
        config_frame = tk.LabelFrame(top_frame, text="설정", font=('Arial', 12, 'bold'))
        config_frame.pack(fill=tk.X, pady=5)
        
        # 설정 그리드
        config_grid = tk.Frame(config_frame)
        config_grid.pack(fill=tk.X, padx=5, pady=5)
        
        # Excel 폴더 경로
        tk.Label(config_grid, text="Excel 폴더:").grid(row=0, column=0, sticky=tk.W, padx=5, pady=3)
        self.folder_var = tk.StringVar(value=r'C:\ai_x\source\Pks_Develop\N시기별음식URL수집\N월별조회메뉴수집_비중반영')
        folder_entry = tk.Entry(config_grid, textvariable=self.folder_var, width=70)
        folder_entry.grid(row=0, column=1, padx=5, pady=3)
        tk.Button(config_grid, text="찾기", command=self.browse_folder).grid(row=0, column=2, padx=5, pady=3)
        
        # 속성 파일 경로
        tk.Label(config_grid, text="속성 파일:").grid(row=1, column=0, sticky=tk.W, padx=5, pady=3)
        self.properties_var = tk.StringVar(value='병합_레시피_포함매핑적용_380개메뉴포함.xlsx')
        properties_entry = tk.Entry(config_grid, textvariable=self.properties_var, width=70)
        properties_entry.grid(row=1, column=1, padx=5, pady=3)
        tk.Button(config_grid, text="찾기", command=self.browse_properties).grid(row=1, column=2, padx=5, pady=3)
        
        # 옵션
        options_frame = tk.Frame(config_grid)
        options_frame.grid(row=2, column=1, sticky=tk.W, padx=5, pady=3)
        
        self.show_images_var = tk.BooleanVar(value=True)
        tk.Checkbutton(options_frame, text="이미지 미리보기", variable=self.show_images_var).pack(side=tk.LEFT)
        
        self.auto_continue_var = tk.BooleanVar(value=False)
        tk.Checkbutton(options_frame, text="자동 진행 (즉시)", variable=self.auto_continue_var).pack(side=tk.LEFT, padx=20)
        
        self.fast_mode_var = tk.BooleanVar(value=False)
        tk.Checkbutton(options_frame, text="고속 처리 모드", variable=self.fast_mode_var).pack(side=tk.LEFT, padx=20)
        
        self.parallel_mode_var = tk.BooleanVar(value=False)
        tk.Checkbutton(options_frame, text="병렬 처리 (10배 빠름)", variable=self.parallel_mode_var).pack(side=tk.LEFT, padx=20)
        
        # 버튼 프레임
        button_frame = tk.Frame(config_frame)
        button_frame.pack(fill=tk.X, pady=5)
        
        self.start_button = tk.Button(button_frame, text="시작", command=self.start_processing, 
                                     bg='lightgreen', font=('Arial', 12, 'bold'), width=8)
        self.start_button.pack(side=tk.LEFT, padx=5)
        
        self.stop_button = tk.Button(button_frame, text="중지", command=self.stop_processing, 
                                    bg='lightcoral', font=('Arial', 12, 'bold'), width=8, state=tk.DISABLED)
        self.stop_button.pack(side=tk.LEFT, padx=5)
        
        self.continue_button = tk.Button(button_frame, text="다음", command=self.continue_processing, 
                                        bg='lightblue', font=('Arial', 12, 'bold'), width=8, state=tk.DISABLED)
        self.continue_button.pack(side=tk.LEFT, padx=5)
        
        self.save_button = tk.Button(button_frame, text="저장", command=self.save_current, 
                                    bg='lightyellow', font=('Arial', 12, 'bold'), width=8, state=tk.DISABLED)
        self.save_button.pack(side=tk.LEFT, padx=5)
        
        # 상태 표시
        status_frame = tk.Frame(button_frame)
        status_frame.pack(side=tk.RIGHT, padx=10)
        
        self.status_var = tk.StringVar(value="준비됨")
        status_label = tk.Label(status_frame, textvariable=self.status_var, font=('Arial', 11, 'bold'), fg='darkgreen')
        status_label.pack()
        
        self.progress = ttk.Progressbar(status_frame, mode='determinate', length=250)
        self.progress.pack(pady=2)
        
        # 중간 영역: 메인 콘텐츠 (좌우 분할)
        middle_frame = tk.Frame(main_container)
        middle_frame.pack(fill=tk.BOTH, expand=True, pady=5)
        
        # 좌측: 이미지 영역 (창은 작게, 이미지는 크게)
        left_panel = tk.LabelFrame(middle_frame, text="이미지 분석", font=('Arial', 12, 'bold'))
        left_panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 5))
        
        # 이미지 표시 영역 (창 크기는 작게)
        self.image_frame = tk.Frame(left_panel)
        self.image_frame.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)
        
        self.image_label = tk.Label(self.image_frame, text="이미지 분석을 시작하려면 '시작' 버튼을 눌러주세요", 
                                   font=('Arial', 10), fg='gray', 
                                   width=40, height=12, relief=tk.SUNKEN, bg='white')
        self.image_label.pack(expand=True, fill=tk.BOTH)
        
        # 이미지 정보 (하단, 더 컴팩트하게)
        image_info_frame = tk.Frame(left_panel)
        image_info_frame.pack(fill=tk.X, padx=5, pady=2)
        
        # URL 정보
        url_label_frame = tk.LabelFrame(image_info_frame, text="이미지 URL", font=('Arial', 9, 'bold'))
        url_label_frame.pack(fill=tk.X, pady=1)
        
        self.url_text = tk.Text(url_label_frame, height=1, font=('Arial', 8), state=tk.DISABLED)
        self.url_text.pack(fill=tk.X, padx=2, pady=2)
        
        # 예측 결과
        result_label_frame = tk.LabelFrame(image_info_frame, text="예측 결과", font=('Arial', 9, 'bold'))
        result_label_frame.pack(fill=tk.X, pady=1)
        
        self.result_text = tk.Text(result_label_frame, height=2, font=('Arial', 9), state=tk.DISABLED)
        self.result_text.pack(fill=tk.X, padx=2, pady=2)
        
        # 우측: 정보 패널 (더 넓게)
        right_panel = tk.Frame(middle_frame, width=500)  # 더 넓은 고정 폭
        right_panel.pack(side=tk.RIGHT, fill=tk.Y, padx=(5, 0))
        right_panel.pack_propagate(False)  # 크기 고정
        
        # 메뉴 정보 (상단)
        menu_frame = tk.LabelFrame(right_panel, text="메뉴 정보", font=('Arial', 11, 'bold'))
        menu_frame.pack(fill=tk.X, pady=(0, 3))
        
        # 메뉴명 표시
        self.menu_info_text = tk.Text(menu_frame, height=3, font=('Arial', 10), state=tk.DISABLED)
        self.menu_info_text.pack(fill=tk.X, padx=3, pady=3)
        
        # 레시피 정보 (중간)
        recipe_frame = tk.LabelFrame(right_panel, text="레시피 정보", font=('Arial', 11, 'bold'))
        recipe_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 3))
        
        self.recipe_text = scrolledtext.ScrolledText(recipe_frame, wrap=tk.WORD, font=('Arial', 9), state=tk.DISABLED)
        self.recipe_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
        
        # 실행 로그 (하단)
        log_frame = tk.LabelFrame(right_panel, text="실행 로그", font=('Arial', 11, 'bold'))
        log_frame.pack(fill=tk.BOTH, expand=True, pady=(3, 0))
        
        # 로그 컨트롤
        log_control = tk.Frame(log_frame)
        log_control.pack(fill=tk.X, padx=3, pady=1)
        
        tk.Button(log_control, text="지우기", command=self.clear_log, font=('Arial', 8)).pack(side=tk.LEFT)
        tk.Button(log_control, text="저장", command=self.save_log, font=('Arial', 8)).pack(side=tk.LEFT, padx=3)
        
        # 자동 스크롤 옵션
        self.auto_scroll_var = tk.BooleanVar(value=True)
        tk.Checkbutton(log_control, text="자동 스크롤", variable=self.auto_scroll_var, font=('Arial', 8)).pack(side=tk.RIGHT)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, wrap=tk.WORD, font=('Consolas', 8), 
                                                 height=10, state=tk.DISABLED)
        self.log_text.pack(fill=tk.BOTH, expand=True, padx=3, pady=3)
        
        # 하단: 통계 정보
        bottom_frame = tk.LabelFrame(main_container, text="처리 통계", font=('Arial', 12, 'bold'))
        bottom_frame.pack(fill=tk.X, pady=(10, 0))
        
        stats_grid = tk.Frame(bottom_frame)
        stats_grid.pack(fill=tk.X, padx=10, pady=5)
        
        # 통계 라벨들
        self.total_files_var = tk.StringVar(value="파일: 0")
        self.total_processed_var = tk.StringVar(value="처리: 0")
        self.total_success_var = tk.StringVar(value="성공: 0")
        self.success_rate_var = tk.StringVar(value="성공률: 0%")
        self.elapsed_time_var = tk.StringVar(value="시간: 00:00")
        
        tk.Label(stats_grid, textvariable=self.total_files_var, font=('Arial', 10, 'bold')).pack(side=tk.LEFT, padx=10)
        tk.Label(stats_grid, textvariable=self.total_processed_var, font=('Arial', 10, 'bold')).pack(side=tk.LEFT, padx=10)
        tk.Label(stats_grid, textvariable=self.total_success_var, font=('Arial', 10, 'bold')).pack(side=tk.LEFT, padx=10)
        tk.Label(stats_grid, textvariable=self.success_rate_var, font=('Arial', 10, 'bold'), fg='blue').pack(side=tk.LEFT, padx=10)
        tk.Label(stats_grid, textvariable=self.elapsed_time_var, font=('Arial', 10, 'bold'), fg='purple').pack(side=tk.RIGHT, padx=10)
        
        # 키보드 이벤트 바인딩
        self.root.bind('<Key>', self.on_key_press)
        self.root.focus_set()
        
        # 시작 로그 (UI 설정 완료 후)
        self.add_log("음식 이미지 예측 시스템이 준비되었습니다.", "INFO")
        self.add_log("Excel 폴더와 속성 파일 경로를 확인한 후 '시작' 버튼을 눌러주세요.", "INFO")
    
    def on_key_press(self, event):
        """키보드 이벤트 처리"""
        if not self.is_running:
            return
            
        if event.keysym == 'Return' or event.keysym == 'space':
            self.continue_processing()
        elif event.keysym.lower() == 's':
            self.save_current()
        elif event.keysym.lower() == 'q':
            self.stop_processing()
    
    def load_translation_table(self):
        """영어-한국어 음식명 매칭 테이블 로드"""
        # 고정된 매칭 테이블 (50개 항목)
        self.english_to_korean = {
            'BBQ': '바비큐립',
            'baguette': '바게트',
            'banh_mi': '반미',
            'bingsu': '팥빙수',
            'bulgogi': '불고기',
            'bunza': '분짜',
            'burger': '치즈버거',
            'burrito': '부리또',
            'cake': '치즈케이크',
            'chicken': '후라이드치킨',
            'cookie': '쿠키',
            'croissant': '크루아상',
            'croque_monsieur': '크로크무슈',
            'curry': '레드커리',
            'dim_sum': '딤섬',
            'egg_benedict': '에그베네딕트',
            'french_fries': '감자튀김',
            'french_toast': '프렌치토스트',
            'galbi': '돼지갈비',
            'gimbap': '참치김밥',
            'gratin': '그라탱',
            'hot_pot': '훠궈',
            'jajangmyeon': '짜장면',
            'japchae': '잡채밥',
            'kebap': '케밥',
            'kimchi_stew': '김치찌개',
            'korean_pancake': '해물전',
            'lasana': '라자냐',
            'macaroon': '마카롱',
            'mapa_tofu': '마파두부밥',
            'muffin': '머핀',
            'nachos': '나초',
            'pad_thai': '팟타이',
            'pan_cake': '팬케이크',
            'pasta': '마라파스타',
            'pizza': '한국식피자',
            'quesadilla': '케사디아',
            'ramen': '라멘',
            'rice_noodle': '쌀국수',
            'risotto': '해물리조또',
            'salad': '시저샐러드',
            'sashimi': '생선회',
            'seaweed_soup': '미역국',
            'soba': '소바',
            'soup': '크림수프',
            'steak': '스테이크와감자',
            'sushi': '연어초밥',
            'takoyaki': '타코야키',
            'tteokbokki': '국물떡볶이',
            'udon': '우동'
        }
        
        self.add_log(f"매칭 테이블 로드 완료: {len(self.english_to_korean)}개 항목", "SUCCESS")
    
    def translate_food_name(self, english_name):
        """영어 음식명을 한국어로 변환"""
        if not english_name:
            return english_name
        
        # 정확한 매칭 시도
        if english_name in self.english_to_korean:
            return self.english_to_korean[english_name]
        
        # 부분 매칭 시도 (언더스코어 등 고려)
        english_clean = english_name.replace('_', ' ').lower()
        for eng_key, kor_val in self.english_to_korean.items():
            eng_key_clean = eng_key.replace('_', ' ').lower()
            if english_clean == eng_key_clean:
                return kor_val
        
        # 매칭되지 않으면 원본 반환
        return english_name
    
    def browse_folder(self):
        """폴더 선택"""
        folder = filedialog.askdirectory(title="Excel 파일이 있는 폴더 선택")
        if folder:
            self.folder_var.set(folder)
            self.add_log(f"Excel 폴더 선택: {folder}", "INFO")
    
    def browse_properties(self):
        """속성 파일 선택"""
        file = filedialog.askopenfilename(
            title="레시피 속성 파일 선택",
            filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
        )
        if file:
            self.properties_var.set(file)
            self.add_log(f"속성 파일 선택: {file}", "INFO")
    
    def add_log(self, message, level="INFO"):
        """로그 메시지 추가"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        # 레벨별 색상 설정
        color_map = {
            "INFO": "black",
            "SUCCESS": "green", 
            "WARNING": "orange",
            "ERROR": "red",
            "STEP": "blue",
            "PROCESS": "purple"
        }
        
        color = color_map.get(level, "black")
        
        self.log_text.config(state=tk.NORMAL)
        self.log_text.insert(tk.END, f"[{timestamp}] [{level}] {message}\n")
        
        # 마지막 라인에 색상 적용
        line_start = self.log_text.index("end-2l linestart")
        line_end = self.log_text.index("end-1l lineend")
        self.log_text.tag_add(level, line_start, line_end)
        self.log_text.tag_config(level, foreground=color)
        
        self.log_text.config(state=tk.DISABLED)
        
        # 자동 스크롤
        if self.auto_scroll_var.get():
            self.log_text.see(tk.END)
        
        # UI 업데이트
        self.root.update()
    
    def clear_log(self):
        """로그 지우기"""
        self.log_text.config(state=tk.NORMAL)
        self.log_text.delete(1.0, tk.END)
        self.log_text.config(state=tk.DISABLED)
        self.add_log("로그가 지워졌습니다.", "INFO")
    
    def save_log(self):
        """로그 파일로 저장"""
        try:
            filename = filedialog.asksaveasfilename(
                defaultextension=".txt",
                filetypes=[("Text files", "*.txt"), ("All files", "*.*")],
                title="로그 저장",
                initialname=f"prediction_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
            )
            if filename:
                with open(filename, 'w', encoding='utf-8') as f:
                    f.write(self.log_text.get(1.0, tk.END))
                self.add_log(f"로그 저장 완료: {filename}", "SUCCESS")
        except Exception as e:
            self.add_log(f"로그 저장 실패: {e}", "ERROR")
    
    def download_images_parallel(self, urls, max_workers=10):
        """이미지들을 병렬로 다운로드"""
        results = {}
        
        def download_single(url_idx_pair):
            idx, url = url_idx_pair
            try:
                response = requests.get(url, timeout=10)
                response.raise_for_status()
                
                content_type = response.headers.get('content-type', '')
                if not content_type.startswith('image/'):
                    return idx, None, f"이미지 파일이 아님: {content_type}"
                
                img = Image.open(BytesIO(response.content))
                
                if img.width < 50 or img.height < 50:
                    return idx, None, f"이미지가 너무 작음: {img.width}x{img.height}"
                
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                return idx, img, "성공"
                
            except requests.exceptions.Timeout:
                return idx, None, "타임아웃"
            except requests.exceptions.RequestException as e:
                return idx, None, f"네트워크 오류: {str(e)[:50]}"
            except Exception as e:
                return idx, None, f"이미지 처리 오류: {str(e)[:50]}"
        
        # 병렬 다운로드 실행
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_idx = {executor.submit(download_single, (idx, url)): idx for idx, url in enumerate(urls)}
            
            for future in as_completed(future_to_idx):
                idx, img, msg = future.result()
                results[idx] = (img, msg)
        
        return results
    
    def predict_images_batch(self, images, batch_size=16):
        """이미지들을 배치로 예측 - GPU 최적화"""
        if not images:
            return []
        
        try:
            # GPU 사용 시 더 큰 배치 크기 사용
            if GPU_AVAILABLE:
                effective_batch_size = min(batch_size * 2, 32)  # GPU에서 더 큰 배치
                device = '/GPU:0'
                self.add_log(f"GPU 배치 예측 시작 (배치 크기: {effective_batch_size})", "INFO")
            else:
                effective_batch_size = min(batch_size, 8)  # CPU에서 작은 배치
                device = '/CPU:0'
                self.add_log(f"CPU 배치 예측 시작 (배치 크기: {effective_batch_size})", "INFO")
            
            # 이미지를 배치 단위로 나누어 처리
            all_results = []
            
            for i in range(0, len(images), effective_batch_size):
                batch_images = images[i:i + effective_batch_size]
                
                with tf.device(device):
                    # 이미지 전처리 배치
                    batch_tensors = []
                    for img in batch_images:
                        img_array = np.array(img)
                        img_resized = tf.image.resize(img_array, IMG_SIZE)
                        batch_tensors.append(img_resized)
                    
                    # 배치 텐서 생성
                    batch_tensor = tf.stack(batch_tensors)
                    
                    # 배치 예측 (GPU에서 실행)
                    inp_eff = eff_pre(batch_tensor)
                    inp_res = res_pre(batch_tensor)
                    
                    # GPU에서 모델 예측
                    p_eff = self.eff_model.predict(inp_eff, verbose=0, batch_size=effective_batch_size)
                    p_res = self.res_model.predict(inp_res, verbose=0, batch_size=effective_batch_size)
                    p_cnn = (p_eff + p_res) / 2.0
                    
                    # XGBoost 배치 예측 (CPU에서 실행)
                    if self.xgb_model:
                        try:
                            with tf.device('/CPU:0'):  # XGBoost는 CPU에서
                                feat_eff = Model(self.eff_model.input, self.eff_model.get_layer('gap').output).predict(inp_eff, verbose=0)
                                feat_res = Model(self.res_model.input, self.res_model.get_layer('gap').output).predict(inp_res, verbose=0)
                                feat_batch = np.hstack([feat_eff, feat_res])
                                p_xgb = self.xgb_model.predict_proba(feat_batch)
                            
                            ensemble = p_cnn * 0.6 + p_xgb * 0.4
                        except Exception as e:
                            self.add_log(f"XGBoost 예측 실패, CNN만 사용: {e}", "WARNING")
                            ensemble = p_cnn
                    else:
                        ensemble = p_cnn
                    
                    # 각 이미지별 상위 3개 결과 추출
                    for j in range(len(batch_images)):
                        img_probs = ensemble[j]
                        top3_indices = np.argsort(img_probs)[-3:][::-1]
                        
                        results = []
                        for rank, idx in enumerate(top3_indices):
                            food_name = self.index_to_label[idx]
                            confidence = float(img_probs[idx])
                            results.append({
                                'rank': rank + 1,
                                'food': food_name,
                                'confidence': confidence
                            })
                        
                        all_results.append(results)
            
            return all_results
            
        except Exception as e:
            self.add_log(f"배치 예측 실패: {e}, 개별 예측으로 폴백", "ERROR")
            # 배치 예측 실패시 개별 예측으로 폴백
            individual_results = []
            for img in images:
                result, _ = self.predict_image(img)
                individual_results.append(result if result else [])
            return individual_results
    
    def show_large_image(self, pil_image):
        """URL 배치를 병렬로 처리"""
        # 1. 이미지 병렬 다운로드
        download_results = self.download_images_parallel(url_batch)
        
        # 2. 성공한 이미지들만 추출
        successful_images = []
        successful_indices = []
        successful_menus = []
        
        for i, (img, msg) in download_results.items():
            if img is not None:
                successful_images.append(img)
                successful_indices.append(indices_batch[i])
                successful_menus.append(menu_batch[i])
        
        if not successful_images:
            return []
        
        # 3. 배치 예측
        batch_predictions = self.predict_images_batch(successful_images)
        
        # 4. 결과 조합
        results = []
        for i, predictions in enumerate(batch_predictions):
            if predictions:
                results.append({
                    'index': successful_indices[i],
                    'image': successful_images[i],
                    'predictions': predictions,
                    'menu_name': successful_menus[i],
                    'url': url_batch[successful_indices[i]]
                })
        
        return results
        """큰 이미지를 새 창에서 표시"""
        try:
            # 새 창 생성
            image_window = tk.Toplevel(self.root)
            image_window.title("큰 이미지 보기")
            image_window.geometry("1000x700")
            image_window.configure(bg='black')
            
            # 큰 이미지 크기 계산 (최대 크기로)
            max_width = 950
            max_height = 650
            
            img_ratio = pil_image.width / pil_image.height
            window_ratio = max_width / max_height
            
            if img_ratio > window_ratio:
                # 이미지가 더 가로로 길 때
                large_size = (max_width, int(max_width / img_ratio))
            else:
                # 이미지가 더 세로로 길 때
                large_size = (int(max_height * img_ratio), max_height)
            
            # 큰 이미지 생성
            large_img = pil_image.resize(large_size, Image.Resampling.LANCZOS)
            large_photo = ImageTk.PhotoImage(large_img)
            
            # 이미지 라벨
            large_label = tk.Label(image_window, image=large_photo, bg='black')
            large_label.pack(expand=True)
            large_label.image = large_photo  # 참조 유지
            
            # 닫기 버튼
            close_button = tk.Button(image_window, text="닫기", command=image_window.destroy,
                                   font=('Arial', 12), bg='white', fg='black')
            close_button.pack(pady=10)
            
            # ESC 키로 닫기
            image_window.bind('<Escape>', lambda e: image_window.destroy())
            
            # 클릭으로 닫기
            large_label.bind("<Button-1>", lambda e: image_window.destroy())
            
            # 창을 맨 앞으로
            image_window.lift()
            image_window.focus_set()
            
        except Exception as e:
            self.add_log(f"큰 이미지 표시 오류: {e}", "ERROR")
    
    def update_image_display(self, pil_image, url, results, menu_name, properties_info):
        """이미지 및 결과 표시 업데이트"""
        try:
            # 이미지 표시 (창은 작지만 실제 이미지는 크게)
            if pil_image:
                # 창에 맞는 작은 썸네일 크기로 조정
                frame_width = self.image_frame.winfo_width()
                frame_height = self.image_frame.winfo_height()
                
                # 프레임 크기가 아직 설정되지 않은 경우 작은 기본값 사용
                if frame_width <= 1 or frame_height <= 1:
                    thumbnail_size = (300, 200)  # 작은 썸네일 크기
                else:
                    # 프레임 크기에 맞춰 작은 썸네일로 조정
                    max_width = max(frame_width - 20, 200)
                    max_height = max(frame_height - 20, 150)
                    
                    # 원본 비율 유지하면서 작은 크기로
                    img_ratio = pil_image.width / pil_image.height
                    frame_ratio = max_width / max_height
                    
                    if img_ratio > frame_ratio:
                        thumbnail_size = (max_width, int(max_width / img_ratio))
                    else:
                        thumbnail_size = (int(max_height * img_ratio), max_height)
                
                # 작은 썸네일 생성 (창에 표시용)
                img_thumbnail = pil_image.resize(thumbnail_size, Image.Resampling.LANCZOS)
                photo = ImageTk.PhotoImage(img_thumbnail)
                
                self.image_label.config(image=photo, text="")
                self.image_label.image = photo  # 참조 유지
                
                # 클릭하면 큰 이미지를 새 창에서 보여주는 이벤트 추가
                self.image_label.bind("<Button-1>", lambda e: self.show_large_image(pil_image))
            
            # URL 표시
            self.url_text.config(state=tk.NORMAL)
            self.url_text.delete(1.0, tk.END)
            self.url_text.insert(tk.END, url)
            self.url_text.config(state=tk.DISABLED)
            
            # 예측 결과 표시 (한국어로 변환)
            self.result_text.config(state=tk.NORMAL)
            self.result_text.delete(1.0, tk.END)
            
            if results:
                for result in results:
                    rank_display = ["1위", "2위", "3위"][result['rank'] - 1]
                    english_name = result['food']
                    korean_name = self.translate_food_name(english_name)
                    
                    # 한국어 번역이 있으면 표시
                    if korean_name != english_name:
                        result_line = f"{rank_display}: {korean_name} ({english_name}) (신뢰도: {result['confidence']:.3f})\n"
                    else:
                        result_line = f"{rank_display}: {english_name} (신뢰도: {result['confidence']:.3f})\n"
                    
                    self.result_text.insert(tk.END, result_line)
            
            self.result_text.config(state=tk.DISABLED)
            
            # 메뉴 정보 표시 (한국어 번역 포함)
            self.menu_info_text.config(state=tk.NORMAL)
            self.menu_info_text.delete(1.0, tk.END)
            
            menu_info_content = ""
            if menu_name:
                menu_info_content += f"시트 메뉴명: {menu_name}\n"
            if results:
                english_prediction = results[0]['food']
                korean_prediction = self.translate_food_name(english_prediction)
                
                if korean_prediction != english_prediction:
                    menu_info_content += f"예측 결과: {korean_prediction} ({english_prediction})\n"
                else:
                    menu_info_content += f"예측 결과: {english_prediction}\n"
                
                menu_info_content += f"신뢰도: {results[0]['confidence']:.3f}\n"
                
                # 매칭 상태
                if menu_name and (korean_prediction in menu_name or english_prediction in menu_name):
                    menu_info_content += "매칭 상태: 일치\n"
                elif menu_name:
                    menu_info_content += "매칭 상태: 불일치\n"
                else:
                    menu_info_content += "매칭 상태: 메뉴명 없음\n"
            
            self.menu_info_text.insert(tk.END, menu_info_content)
            self.menu_info_text.config(state=tk.DISABLED)
            
            # 레시피 정보 표시 (시트 메뉴명 우선 사용)
            self.recipe_text.config(state=tk.NORMAL)
            self.recipe_text.delete(1.0, tk.END)
            
            # 시트의 메뉴명을 우선적으로 사용
            search_menu = menu_name if menu_name else (results[0]['food'] if results else None)
            
            if search_menu and properties_info:
                found_menu = properties_info['menu_name']
                properties = properties_info['properties']
                
                recipe_content = f"레시피 기준 메뉴: {found_menu}\n"
                if found_menu != search_menu:
                    recipe_content += f"검색에 사용된 메뉴: {search_menu}\n"
                recipe_content += "\n"
                
                sheet_names = {
                    '주요식재료': '주요 식재료',
                    '양념소스': '양념 및 소스', 
                    '육수베이스': '육수/베이스',
                    '조리순서': '조리 순서',
                    '조리정보': '조리 정보',
                    '특징맛': '맛 특성',
                    '영양정보': '영양 정보',
                    '서빙조리팁': '조리 팁'
                }
                
                for sheet_name, sheet_data in properties.items():
                    sheet_title = sheet_names.get(sheet_name, sheet_name)
                    recipe_content += f"\n--- {sheet_title} ---\n"
                    
                    count = 0
                    for ingredient, amount in sheet_data.items():
                        if count >= 15:  # 최대 15개까지
                            recipe_content += "  ... (더 많은 항목 있음)\n"
                            break
                        recipe_content += f"  • {ingredient}: {amount}\n"
                        count += 1
                    
                    recipe_content += "\n"
            else:
                recipe_content = "레시피 정보를 찾을 수 없습니다.\n\n"
                if search_menu:
                    recipe_content += f"검색한 메뉴: '{search_menu}'\n"
                    recipe_content += "속성 파일에서 일치하는 메뉴를 찾을 수 없습니다.\n"
                    recipe_content += "메뉴명이 정확히 일치하지 않을 수 있습니다."
                else:
                    recipe_content += "메뉴명 정보가 없습니다."
            
            self.recipe_text.insert(tk.END, recipe_content)
            self.recipe_text.config(state=tk.DISABLED)
            
        except Exception as e:
            self.add_log(f"화면 업데이트 오류: {e}", "ERROR")
    
    def update_stats(self, total_files=0, total_processed=0, total_success=0, start_time=None):
        """통계 정보 업데이트"""
        self.total_files_var.set(f"파일: {total_files}")
        self.total_processed_var.set(f"처리: {total_processed}")
        self.total_success_var.set(f"성공: {total_success}")
        
        if total_processed > 0:
            success_rate = (total_success / total_processed) * 100
            self.success_rate_var.set(f"성공률: {success_rate:.1f}%")
        else:
            self.success_rate_var.set("성공률: 0%")
        
        if start_time:
            elapsed = time.time() - start_time
            hours = int(elapsed // 3600)
            minutes = int((elapsed % 3600) // 60)
            seconds = int(elapsed % 60)
            if hours > 0:
                time_str = f"시간: {hours:02d}:{minutes:02d}:{seconds:02d}"
            else:
                time_str = f"시간: {minutes:02d}:{seconds:02d}"
            self.elapsed_time_var.set(time_str)
    
    def start_processing(self):
        """처리 시작"""
        if self.is_running:
            return
        
        # 기본 검증
        folder_path = self.folder_var.get()
        properties_file = self.properties_var.get()
        
        if not os.path.exists(folder_path):
            messagebox.showerror("오류", f"Excel 폴더가 존재하지 않습니다:\n{folder_path}")
            return
        
        if not os.path.exists('models'):
            messagebox.showerror("오류", "models 폴더가 존재하지 않습니다.")
            return
        
        self.is_running = True
        self.stop_requested = False
        self.user_action = None
        
        # 버튼 상태 변경
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        
        # 진행률 바 시작
        self.progress.config(mode='indeterminate')
        self.progress.start()
        
        # 별도 스레드에서 처리 실행
        thread = threading.Thread(target=self.run_processing)
        thread.daemon = True
        thread.start()
    
    def stop_processing(self):
        """처리 중지"""
        self.stop_requested = True
        self.user_action = 'quit'
        self.add_log("사용자가 중지를 요청했습니다.", "WARNING")
    
    def continue_processing(self):
        """다음으로 진행"""
        self.user_action = 'continue'
        self.continue_button.config(state=tk.DISABLED)
        self.save_button.config(state=tk.DISABLED)
    
    def save_current(self):
        """현재 결과 저장"""
        if self.current_image and self.current_results:
            try:
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f"prediction_{self.current_results[0]['food']}_{timestamp}.png"
                self.current_image.save(filename)
                self.add_log(f"이미지 저장 완료: {filename}", "SUCCESS")
            except Exception as e:
                self.add_log(f"저장 실패: {e}", "ERROR")
        
        self.user_action = 'save'
        self.continue_button.config(state=tk.DISABLED)
        self.save_button.config(state=tk.DISABLED)
    
    def wait_for_user_action(self):
        """사용자 액션 대기"""
        if not self.show_images_var.get():
            return 'continue'
        
        # 버튼 활성화
        self.continue_button.config(state=tk.NORMAL)
        self.save_button.config(state=tk.NORMAL)
        
        # 자동 진행 옵션 (대기시간 제거)
        if self.auto_continue_var.get():
            self.add_log("자동 진행 모드 - 즉시 다음으로 진행", "INFO")
            self.user_action = 'continue'  # 즉시 진행, 대기시간 없음
        else:
            self.add_log("사용자 입력 대기 중... (Enter/Space: 다음, S: 저장, Q: 종료)", "INFO")
            
            # 사용자 입력 대기
            while not self.user_action and not self.stop_requested:
                time.sleep(0.1)
                self.root.update()
        
        action = self.user_action
        self.user_action = None
        return action if action else 'quit'
    
    def run_processing(self):
        """실제 처리 실행 (별도 스레드)"""
        start_time = time.time()
        
        try:
            self.add_log("처리 시작", "STEP")
            
            # 예측기 초기화
            self.status_var.set("예측기 초기화 중...")
            self.add_log("예측기 초기화 중...", "STEP")
            
            self.predictor = EnhancedFoodPredictor(
                properties_file=self.properties_var.get(),
                log_callback=self.add_log
            )
            
            # Excel 파일 처리
            self.status_var.set("Excel 파일 처리 중...")
            self.add_log("Excel 파일 처리 시작", "STEP")
            
            self.process_excel_files(start_time)
            
        except Exception as e:
            self.add_log(f"처리 중 오류 발생: {e}", "ERROR")
            messagebox.showerror("오류", f"처리 중 오류가 발생했습니다:\n{e}")
        finally:
            self.is_running = False
            self.start_button.config(state=tk.NORMAL)
            self.stop_button.config(state=tk.DISABLED)
            self.continue_button.config(state=tk.DISABLED)
            self.save_button.config(state=tk.DISABLED)
            self.progress.stop()
            self.progress.config(mode='determinate', value=0)
            
            if self.stop_requested:
                self.status_var.set("사용자 중지")
                self.add_log("사용자 요청으로 처리가 중단되었습니다.", "WARNING")
            else:
                self.status_var.set("처리 완료")
                self.add_log("모든 처리가 완료되었습니다!", "SUCCESS")
    
    def process_excel_files(self, start_time):
        """Excel 파일들 처리"""
        folder_path = self.folder_var.get()
        
        # Excel 파일 찾기
        excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))
        self.add_log(f"Excel 파일 {len(excel_files)}개 발견", "INFO")
        
        if not excel_files:
            self.add_log("처리할 Excel 파일이 없습니다.", "WARNING")
            return
        
        total_processed = 0
        total_success = 0
        
        for file_idx, excel_file in enumerate(excel_files, 1):
            if self.stop_requested:
                break
            
            self.add_log(f"[{file_idx}/{len(excel_files)}] 파일 처리: {os.path.basename(excel_file)}", "STEP")
            self.update_stats(len(excel_files), total_processed, total_success, start_time)
            
            try:
                # Excel 읽기
                xl_file = pd.ExcelFile(excel_file)
                sheet_names = [name for name in xl_file.sheet_names if '요약' not in name]
                
                for sheet_idx, sheet_name in enumerate(sheet_names, 1):
                    if self.stop_requested:
                        break
                    
                    self.add_log(f"[{sheet_idx}/{len(sheet_names)}] 시트 처리: {sheet_name}", "PROCESS")
                    
                    df = pd.read_excel(excel_file, sheet_name=sheet_name)
                    
                    # URL 컬럼 찾기
                    url_col = None
                    for col in df.columns:
                        if any(keyword in col.lower() for keyword in ['url', 'image', '이미지']):
                            url_col = col
                            break
                    
                    if not url_col:
                        self.add_log(f"URL 컬럼을 찾을 수 없습니다: {list(df.columns)}", "WARNING")
                        continue
                    
                    # 메뉴명 컬럼 찾기 (시트에서)
                    menu_col = None
                    menu_col_candidates = ['상세메뉴', '메뉴명', '메뉴', 'menu', 'Menu', 'MENU']
                    for col in df.columns:
                        for candidate in menu_col_candidates:
                            if candidate in col:
                                menu_col = col
                                break
                        if menu_col:
                            break
                    
                    if menu_col:
                        self.add_log(f"메뉴명 컬럼 발견: {menu_col}", "INFO")
                    else:
                        self.add_log("메뉴명 컬럼을 찾을 수 없습니다. 예측 결과만 사용합니다.", "WARNING")
                    
                    # 결과 컬럼들 추가 (한국어 + 영어)
                    df['예측_음식명'] = ''  # 한국어 예측 결과
                    df['예측_신뢰도'] = 0.0
                    df['예측_2위'] = ''  # 한국어 2위
                    df['예측_3위'] = ''  # 한국어 3위
                    df['예측_영어명'] = ''  # 영어 원본 결과
                    df['예측_영어_2위'] = ''  # 영어 2위
                    df['예측_영어_3위'] = ''  # 영어 3위
                    
                    valid_urls = df[~df[url_col].isna() & (df[url_col] != '')].shape[0]
                    self.add_log(f"유효한 URL: {valid_urls}개", "INFO")
                    
                    if valid_urls == 0:
                        continue
                    
                    sheet_processed = 0
                    sheet_success = 0
                    
                    for idx, row in df.iterrows():
                        if self.stop_requested:
                            break
                        
                        url = row[url_col]
                        if pd.isna(url) or str(url).strip() == '':
                            continue
                        
                        # 시트의 메뉴명 가져오기
                        sheet_menu_name = None
                        if menu_col and not pd.isna(row[menu_col]):
                            sheet_menu_name = str(row[menu_col]).strip()
                        
                        sheet_processed += 1
                        total_processed += 1
                        
                        progress_info = f"[{sheet_processed}/{valid_urls}] {sheet_name}"
                        self.status_var.set(f"처리 중: {progress_info}")
                        
                        # 진행률 업데이트
                        progress_percent = (sheet_processed / valid_urls) * 100
                        self.progress.config(mode='determinate', value=progress_percent)
                        
                        self.add_log(f"[{sheet_processed:3d}/{valid_urls}] 이미지 분석 중...", "PROCESS")
                        if sheet_menu_name:
                            self.add_log(f"    시트 메뉴명: {sheet_menu_name}", "PROCESS")
                        
                        # 이미지 다운로드
                        pil_image, download_msg = self.predictor.download_image(url)
                        
                        if pil_image is None:
                            self.add_log(f"다운로드 실패: {download_msg}", "WARNING")
                            continue
                        
                        # 이미지 예측
                        results, predict_msg = self.predictor.predict_image(pil_image)
                        
                        if results is None:
                            self.add_log(f"예측 실패: {predict_msg}", "WARNING")
                            continue
                        
                        # 성공 - 결과 저장 (한국어로 변환하여 저장)
                        sheet_success += 1
                        total_success += 1
                        
                        # 예측 결과를 한국어로 변환
                        english_prediction = results[0]['food']
                        korean_prediction = self.translate_food_name(english_prediction)
                        
                        df.loc[idx, '예측_음식명'] = korean_prediction
                        df.loc[idx, '예측_신뢰도'] = round(results[0]['confidence'], 3)
                        if len(results) > 1:
                            df.loc[idx, '예측_2위'] = self.translate_food_name(results[1]['food'])
                        if len(results) > 2:
                            df.loc[idx, '예측_3위'] = self.translate_food_name(results[2]['food'])
                        
                        # 원본 영어 결과도 별도 컬럼에 저장
                        df.loc[idx, '예측_영어명'] = english_prediction
                        if len(results) > 1:
                            df.loc[idx, '예측_영어_2위'] = results[1]['food']
                        if len(results) > 2:
                            df.loc[idx, '예측_영어_3위'] = results[2]['food']
                        
                        self.add_log(f"예측 성공: {korean_prediction} ({english_prediction}) (신뢰도: {results[0]['confidence']:.3f})", "SUCCESS")
                        
                        # 레시피 정보 조회 (시트 메뉴명 우선 사용)
                        search_menu = sheet_menu_name if sheet_menu_name else results[0]['food']
                        properties_info = self.predictor.get_menu_properties(search_menu)
                        
                        # 현재 결과 저장 (저장 기능용)
                        self.current_image = pil_image
                        self.current_results = results
                        self.current_properties = properties_info
                        self.current_menu_name = sheet_menu_name
                        
                        # 화면 업데이트
                        self.update_image_display(pil_image, url, results, sheet_menu_name, properties_info)
                        self.update_stats(len(excel_files), total_processed, total_success, start_time)
                        
                        # 사용자 액션 대기 (이미지 표시 옵션이 켜진 경우)
                        if self.show_images_var.get():
                            user_action = self.wait_for_user_action()
                            
                            if user_action == 'quit':
                                self.stop_requested = True
                                break
                            elif user_action == 'save':
                                self.add_log("이미지 저장됨", "INFO")
                    
                    # 중간 저장
                    if sheet_processed > 0:
                        try:
                            with pd.ExcelWriter(excel_file, mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
                                df.to_excel(writer, sheet_name=sheet_name, index=False)
                            self.add_log(f"시트 '{sheet_name}' 저장 완료", "SUCCESS")
                        except Exception as e:
                            self.add_log(f"저장 실패: {e}", "ERROR")
                    
                    # 시트 완료 통계
                    if sheet_processed > 0:
                        sheet_success_rate = (sheet_success / sheet_processed) * 100
                        self.add_log(f"시트 완료 - 처리: {sheet_processed}, 성공: {sheet_success} ({sheet_success_rate:.1f}%)", "INFO")
            
            except Exception as e:
                self.add_log(f"파일 처리 중 오류: {e}", "ERROR")
        
        # 최종 통계
        self.update_stats(len(excel_files), total_processed, total_success, start_time)
        
        if total_processed > 0:
            overall_success_rate = (total_success / total_processed) * 100
            self.add_log(f"최종 결과 - 총 처리: {total_processed}, 성공: {total_success}, 성공률: {overall_success_rate:.1f}%", "SUCCESS")
    
    def run(self):
        """GUI 실행"""
        self.root.mainloop()

class EnhancedFoodPredictor:
    """음식 이미지 예측기"""
    
    def __init__(self, model_dir='models', timestamp='20250716_144252', properties_file=None, log_callback=None):
        self.model_dir = model_dir
        self.timestamp = timestamp
        self.properties_file = properties_file
        self.properties_data = None
        self.log = log_callback if log_callback else print
        
        self.log("예측기 초기화 시작", "STEP")
        self.load_models()
        self.load_properties()
        self.log("예측기 초기화 완료", "SUCCESS")
    
    def build_model_gpu_optimized(self, base_cls, num_classes):
        """GPU 최적화된 모델 구조 빌드"""
        # GPU에서 모델 빌드
        base = base_cls(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
        
        # GPU 메모리 효율성을 위한 설정
        x = GlobalAveragePooling2D(name='gap')(base.output)
        x = Dense(256, activation='relu')(x)
        x = Dropout(0.2)(x)
        out = Dense(num_classes, activation='softmax', dtype='float32')(x)
        
        model = Model(inputs=base.input, outputs=out)
        
        # GPU 최적화 컴파일
        if GPU_AVAILABLE:
            # GPU에서 실행할 때 mixed precision 사용 (선택적)
            try:
                model.compile(
                    optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy']
                )
            except:
                pass
        
        return model
    
    def load_models(self):
        """모델 로드 - GPU 최적화"""
        try:
            self.log("모델 파일 로드 중...", "INFO")
            
            # GPU 상태 확인
            if GPU_AVAILABLE:
                self.log("GPU 모드로 모델 로드 중...", "SUCCESS")
                # GPU에서 모델 로드
                with tf.device('/GPU:0'):
                    self._load_models_on_device()
            else:
                self.log("CPU 모드로 모델 로드 중...", "WARNING")
                # CPU에서 모델 로드
                with tf.device('/CPU:0'):
                    self._load_models_on_device()
            
        except Exception as e:
            self.log(f"모델 로드 실패: {e}", "ERROR")
            raise
    
    def _load_models_on_device(self):
        """실제 모델 로드 작업"""
        # 라벨 매핑
        label_map_path = f"{self.model_dir}/label_to_index_{self.timestamp}.joblib"
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f"라벨 매핑 파일이 없습니다: {label_map_path}")
        
        label_map = joblib.load(label_map_path)
        self.index_to_label = {v: k for k, v in label_map.items()}
        num_classes = len(label_map)
        self.log(f"라벨 매핑 로드 완료 - 클래스 수: {num_classes}", "SUCCESS")
        
        # EfficientNet (GPU 최적화)
        self.log("EfficientNet 모델 로드 중...", "INFO")
        self.eff_model = self.build_model_gpu_optimized(EfficientNetB0, num_classes)
        eff_weights_path = f"{self.model_dir}/effnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(eff_weights_path):
            raise FileNotFoundError(f"EfficientNet 가중치 파일이 없습니다: {eff_weights_path}")
        self.eff_model.load_weights(eff_weights_path)
        self.log("EfficientNet 모델 로드 완료", "SUCCESS")
        
        # ResNet (GPU 최적화)
        self.log("ResNet 모델 로드 중...", "INFO")
        self.res_model = self.build_model_gpu_optimized(ResNet50V2, num_classes)
        res_weights_path = f"{self.model_dir}/resnet_model_best_{self.timestamp}.h5"
        if not os.path.exists(res_weights_path):
            raise FileNotFoundError(f"ResNet 가중치 파일이 없습니다: {res_weights_path}")
        self.res_model.load_weights(res_weights_path)
        self.log("ResNet 모델 로드 완료", "SUCCESS")
        
        # XGBoost (CPU에서 실행)
        try:
            self.log("XGBoost 모델 로드 중...", "INFO")
            xgb_path = f"{self.model_dir}/xgb_model_{self.timestamp}.joblib"
            if not os.path.exists(xgb_path):
                raise FileNotFoundError(f"XGBoost 파일이 없습니다: {xgb_path}")
            
            # XGBoost는 CPU에서 실행
            with tf.device('/CPU:0'):
                self.xgb_model = joblib.load(xgb_path)
            
            # 호환성 설정
            if hasattr(self.xgb_model, '_Booster'):
                if not hasattr(self.xgb_model, 'use_label_encoder'):
                    self.xgb_model.use_label_encoder = False
                if not hasattr(self.xgb_model, 'eval_metric'):
                    self.xgb_model.eval_metric = 'logloss'
            
            self.log("XGBoost 모델 로드 완료", "SUCCESS")
            
        except Exception as e:
            self.xgb_model = None
            self.log(f"XGBoost 로드 실패: {e} (CNN만 사용)", "WARNING")
    
    def load_properties(self):
        """속성 파일 로드"""
        if not self.properties_file:
            self.properties_file = "병합_레시피_포함매핑적용_380개메뉴포함.xlsx"
        
        try:
            if not os.path.exists(self.properties_file):
                self.log(f"속성 파일이 없습니다: {self.properties_file}", "WARNING")
                return
            
            self.log(f"속성 파일 로드 중: {self.properties_file}", "INFO")
            
            xl_file = pd.ExcelFile(self.properties_file)
            self.properties_data = {}
            
            for sheet_name in xl_file.sheet_names:
                try:
                    df = pd.read_excel(self.properties_file, sheet_name=sheet_name, index_col=0)
                    self.properties_data[sheet_name] = df
                    self.log(f"'{sheet_name}' 시트 로드 완료", "INFO")
                except Exception as e:
                    self.log(f"'{sheet_name}' 시트 로드 실패: {e}", "WARNING")
            
            if self.properties_data:
                total_menus = len(self.properties_data['주요식재료'].columns) if '주요식재료' in self.properties_data else 0
                self.log(f"속성 파일 로드 완료 - 총 {total_menus}개 메뉴", "SUCCESS")
                
        except Exception as e:
            self.log(f"속성 파일 로드 실패: {e}", "ERROR")
            self.properties_data = None
    
    def get_menu_properties(self, menu_name):
        """메뉴명으로 속성 정보 조회"""
        if not self.properties_data or not menu_name:
            return None
        
        try:
            properties = {}
            found_menu = None
            
            # 정확한 매칭 시도
            for sheet_name, df in self.properties_data.items():
                if menu_name in df.columns:
                    found_menu = menu_name
                    menu_data = df[menu_name]
                    
                    sheet_properties = {}
                    for ingredient, value in menu_data.items():
                        if pd.notna(value) and str(value).strip() not in ['', '-', '0']:
                            sheet_properties[ingredient] = str(value)
                    
                    if sheet_properties:
                        properties[sheet_name] = sheet_properties
            
            # 부분 매칭 시도 (정확한 매칭이 없는 경우)
            if not found_menu:
                for sheet_name, df in self.properties_data.items():
                    for col in df.columns:
                        # 더 유연한 매칭: 한쪽이 다른 쪽에 포함되는 경우
                        if menu_name in col or col in menu_name:
                            found_menu = col
                            menu_data = df[col]
                            
                            sheet_properties = {}
                            for ingredient, value in menu_data.items():
                                if pd.notna(value) and str(value).strip() not in ['', '-', '0']:
                                    sheet_properties[ingredient] = str(value)
                            
                            if sheet_properties:
                                properties[sheet_name] = sheet_properties
                            break
                    if found_menu:
                        break
            
            return {'menu_name': found_menu, 'properties': properties} if properties else None
                
        except Exception as e:
            self.log(f"속성 조회 실패: {e}", "ERROR")
            return None
    
    def download_image(self, url):
        """URL에서 이미지 다운로드"""
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            content_type = response.headers.get('content-type', '')
            if not content_type.startswith('image/'):
                return None, f"이미지 파일이 아님: {content_type}"
            
            img = Image.open(BytesIO(response.content))
            
            if img.width < 50 or img.height < 50:
                return None, f"이미지가 너무 작음: {img.width}x{img.height}"
            
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            return img, "성공"
            
        except requests.exceptions.Timeout:
            return None, "타임아웃"
        except requests.exceptions.RequestException as e:
            return None, f"네트워크 오류: {str(e)[:50]}"
        except Exception as e:
            return None, f"이미지 처리 오류: {str(e)[:50]}"
    
    def predict_image(self, pil_image):
        """단일 이미지 예측 - GPU 최적화"""
        try:
            # 디바이스 선택
            device = '/GPU:0' if GPU_AVAILABLE else '/CPU:0'
            
            with tf.device(device):
                # 전처리
                img_array = np.array(pil_image)
                img = tf.image.resize(img_array, IMG_SIZE)
                img = tf.expand_dims(img, axis=0)
                
                # CNN 예측 (GPU에서)
                inp_eff = eff_pre(img)
                inp_res = res_pre(img)
                p_eff = self.eff_model.predict(inp_eff, verbose=0)
                p_res = self.res_model.predict(inp_res, verbose=0)
                p_cnn = (p_eff + p_res) / 2.0
                
                # XGBoost 예측 (CPU에서)
                if self.xgb_model:
                    try:
                        with tf.device('/CPU:0'):  # XGBoost는 CPU에서
                            feat_eff = Model(self.eff_model.input, self.eff_model.get_layer('gap').output).predict(inp_eff, verbose=0)
                            feat_res = Model(self.res_model.input, self.res_model.get_layer('gap').output).predict(inp_res, verbose=0)
                            feat = np.hstack([feat_eff, feat_res])
                            p_xgb = self.xgb_model.predict_proba(feat)
                        
                        ensemble = p_cnn * 0.6 + p_xgb * 0.4
                    except:
                        ensemble = p_cnn
                else:
                    ensemble = p_cnn
                
                # 상위 3개 결과
                ensemble = ensemble.flatten()
                top3_indices = np.argsort(ensemble)[-3:][::-1]
                
                results = []
                for i, idx in enumerate(top3_indices):
                    food_name = self.index_to_label[idx]
                    confidence = float(ensemble[idx])
                    results.append({
                        'rank': i + 1,
                        'food': food_name,
                        'confidence': confidence
                    })
                
                return results, "성공"
            
        except Exception as e:
            return None, f"예측 오류: {str(e)[:50]}"

def main():
    """메인 함수"""
    app = FoodPredictorGUI()
    app.run()

if __name__ == '__main__':
    main()


[GPU INFO] GPU를 찾을 수 없습니다. CPU를 사용합니다.
[15:16:14] WARNING: C:/buildkite-agent/builds/buildkite-windows-cpu-autoscaling-group-i-08de971ced8a8cdc6-1/xgboost/xgboost-ci-windows/src/learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.



Exception in thread Thread-5 (run_processing):
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_12224\2259708631.py", line 1092, in process_excel_files
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_12224\2259708631.py", line 401, in add_log
  File "C:\Users\Admin\anaconda3\envs\ml-dl-nlp\lib\tkinter\__init__.py", line 1675, in configure
    return self._configure('configure', cnf, kw)
  File "C:\Users\Admin\anaconda3\envs\ml-dl-nlp\lib\tkinter\__init__.py", line 1665, in _configure
    self.tk.call(_flatten((self._w, cmd)) + self._options(cnf))
RuntimeError: main thread is not in main loop

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_12224\2259708631.py", line 928, in run_processing
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_12224\2259708631.py", line 1133, in process_excel_files
  File "C:\Users\Admin\AppData\Local\Temp\ip